# Deepfake Classifier — Evaluation
Compares the **original** `facebook/dinov2-base` landmark model against the
**fine-tuned** version (saved in `./fine_tuned_model`) on the test split of
`zguo0525/google-landmarks-v2-mini`.

Additionally, runs the full integrated `DeepfakeClassifier` pipeline on a
sample image to validate the end-to-end workflow.

> ⚠️ **Pre-requisite:** Run `deepfake_classifier_finetuning.ipynb` first to produce `./fine_tuned_model`.

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import torch
import numpy as np
from datasets import load_dataset
from transformers import AutoImageProcessor, AutoModelForImageClassification
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from tqdm.notebook import tqdm
from deepfake_classifier import DeepfakeClassifier, compute_metrics

import huggingface_hub
huggingface_hub.login(token="f")

## 1. Load the Landmark Test Dataset

In [ ]:
dataset = load_dataset('zguo0525/google-landmarks-v2-mini')
test_dataset = dataset['test']
label_names = dataset['train'].features['label'].names
num_classes = len(label_names)
print(f"Loaded {len(test_dataset)} test examples across {num_classes} landmark classes.")

## 2. Load Both Landmark Models
We load the original DINOv2 base model (un-fine-tuned) and the fine-tuned
checkpoint to compare their landmark classification accuracy.

In [ ]:
device = torch.device('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

processor = AutoImageProcessor.from_pretrained('facebook/dinov2-base')

print("Loading original DINOv2 base model...")
original_model = AutoModelForImageClassification.from_pretrained(
    'facebook/dinov2-base',
    num_labels=num_classes,
    id2label={str(i): name for i, name in enumerate(label_names)},
    label2id={name: i for i, name in enumerate(label_names)},
    ignore_mismatched_sizes=True
).to(device)
original_model.eval()
print("Original model ready.")

In [ ]:
print("Loading fine-tuned DINOv2 landmark model...")
fine_tuned_model = AutoModelForImageClassification.from_pretrained('./fine_tuned_model').to(device)
fine_tuned_model.eval()
print("Fine-tuned model ready.")

## 3. Quick Sanity Check — Single Sample

In [ ]:
from IPython.display import display

sample = test_dataset[0]
sample_image = sample['image'].convert('RGB')
sample_label = label_names[sample['label']]

display(sample_image.resize((256, 256)))
print(f"Ground Truth: {sample_label}")

def predict_single(model, image):
    inputs = processor(images=image, return_tensors='pt').to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    idx = logits.argmax(-1).item()
    conf = torch.softmax(logits, dim=-1)[0, idx].item()
    return label_names[idx], round(conf, 4)

orig_label, orig_conf = predict_single(original_model, sample_image)
ft_label, ft_conf     = predict_single(fine_tuned_model, sample_image)

print(f"Original model  : {orig_label} (confidence: {orig_conf})")
print(f"Fine-tuned model: {ft_label} (confidence: {ft_conf})")

## 4. Full Test-Set Evaluation
Runs inference for both models across the entire test split.

> 💡 For a quick smoke-test, uncomment `num_samples = 200`.

In [ ]:
num_samples = len(test_dataset)
# num_samples = 200  # uncomment for a quick test

orig_true, orig_pred = [], []
ft_true,   ft_pred   = [], []

for i in tqdm(range(num_samples), desc="Evaluating both models"):
    item = test_dataset[i]
    img  = item['image'].convert('RGB')
    true = label_names[item['label']]

    orig_lbl, _ = predict_single(original_model, img)
    ft_lbl,   _ = predict_single(fine_tuned_model, img)

    orig_true.append(true); orig_pred.append(orig_lbl)
    ft_true.append(true);   ft_pred.append(ft_lbl)

print("Evaluation complete.")

## 5. Results — Original Model

In [ ]:
print("=== Original DINOv2 (un-fine-tuned) ===")
print("\nClassification Report:")
print(classification_report(orig_true, orig_pred, zero_division=0))

## 6. Results — Fine-Tuned Model

In [ ]:
print("=== Fine-Tuned DINOv2 Landmark Model ===")
print("\nClassification Report:")
print(classification_report(ft_true, ft_pred, zero_division=0))

## 7. Side-by-Side Accuracy Comparison

In [ ]:
orig_acc = accuracy_score(orig_true, orig_pred)
ft_acc   = accuracy_score(ft_true,   ft_pred)

print(f"Original model accuracy  : {orig_acc:.4f} ({orig_acc*100:.2f}%)")
print(f"Fine-tuned model accuracy: {ft_acc:.4f} ({ft_acc*100:.2f}%)")
print(f"Improvement              : {(ft_acc - orig_acc)*100:+.2f} pp")

## 8. Integrated `DeepfakeClassifier` — End-to-End Test
Instantiates the full `DeepfakeClassifier` (face + scene + fine-tuned landmark)
and runs it on a single test image to validate the complete pipeline.

In [ ]:
print("Initialising DeepfakeClassifier with the fine-tuned landmark model...")
deepfake_classifier = DeepfakeClassifier(landmark_model_path="./fine_tuned_model")
print("DeepfakeClassifier ready.")

In [ ]:
sample_image = test_dataset[0]['image'].convert('RGB')
display(sample_image.resize((256, 256)))

# Run without the visual classifier gate (will always run all sub-models)
results = deepfake_classifier.predict(sample_image)

print("\n=== Deepfake Classifier Results ===")
print(f"Final Decision          : {results['final_decision']}")
if results['deepfake_analysis']:
    da = results['deepfake_analysis']
    print(f"Deepfake Confidence     : {da['deepfake_confidence']}")
    print(f"Face Analysis           : {da['face_analysis']}")
    print(f"Scene Analysis          : {da['scene_analysis']}")
    print(f"Landmark Analysis       : {da['landmark_analysis']}")